# Task 4 — Statistical Modeling & Risk-Based Pricing

**Objective:** Build and evaluate predictive models that form the core of a dynamic risk-based pricing system.

**Two modelling goals:**
1. **Severity Model** — predict `TotalClaims` for policies with a claim (regression). Metric: RMSE, R².
2. **Claim Probability Model** — predict whether a policy will claim (classification). Metric: Precision, Recall, F1, ROC-AUC.
3. **Premium Framework** — combine both: `Premium = P(claim) × Predicted Severity + Expense Loading + Profit Margin`

**Algorithms:** Linear Regression / Logistic Regression · Random Forest · XGBoost

## 1. Setup

In [1]:
import sys
sys.path.insert(0, '..')
import gc
import warnings; warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
from src.modeling import (
    engineer_features, encode_features,
    prepare_severity_data, prepare_classification_data,
    evaluate_regression, evaluate_classification,
    build_regression_comparison, build_classification_comparison,
    compute_risk_premium,
)
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
SEED = 42

## 2. Data Preparation

### 2.1 Feature Engineering

New features derived from existing columns:
| Feature | Formula | Business rationale |
|---|---|---|
| `vehicle_age` | transaction_year − RegistrationYear | Older vehicles cost more to repair |
| `transaction_year` / `transaction_month_num` | Parsed from TransactionMonth | Captures temporal trends |
| `premium_rate` | CalculatedPremiumPerTerm / SumInsured | Pricing efficiency per unit of cover |
| `capital_utilisation` | CapitalOutstanding / SumInsured | Leverage proxy — outstanding debt fraction |

**Dropped:** ID columns, constants (Language, Country, ItemType), VehicleIntroDate (superseded by vehicle_age), Model (411 unique values).

In [2]:
df_raw = pd.read_csv('../data/insurance_data_cleaned.csv')
df_eng = engineer_features(df_raw)
df_enc = encode_features(df_eng)
print(f'Raw shape      : {df_raw.shape}')
print(f'Encoded shape  : {df_enc.shape}')
print(f'Any nulls      : {df_enc.isnull().sum().sum()}')
df_enc[['vehicle_age', 'premium_rate', 'capital_utilisation']].describe().round(4)

Raw shape      : (999470, 45)
Encoded shape  : (999470, 158)
Any nulls      : 0


,vehicle_age,premium_rate,capital_utilisation
count,999470.0000,999470.0000,9.994700e+05
mean,4.5286,250.8480,5.264779e+05
std,3.2795,729.0758,3.843732e+06
min,0.0000,0.0000,-2.000000e+02
25%,2.0000,0.0005,0.000000e+00
50%,4.0000,0.0008,0.000000e+00
75%,7.0000,0.0051,0.000000e+00
max,28.0000,2500.0000,7.000000e+07


---
## 3. Severity Model (Regression)

**Target:** `log1p(TotalClaims)` for policies where `TotalClaims > 0` (n=2,775).
Log-transforming the target dampens the extreme right-skew (max claim R393,092 vs median R6,140) and stabilises variance across the prediction range.
Predictions are inverse-transformed via `expm1` for evaluation in original Rand scale.

### 3.1 Train / Test Split (80:20)

In [3]:
X_sev_tr, X_sev_te, y_sev_tr, y_sev_te = prepare_severity_data(df_enc, test_size=0.2, random_state=SEED)
print(f'Train: {X_sev_tr.shape}  |  Test: {X_sev_te.shape}')
print(f'Target (log scale) — mean: {y_sev_tr.mean():.2f}  std: {y_sev_tr.std():.2f}')

Train: (2220, 156)  |  Test: (555, 156)
Target (log scale) — mean: 8.94  std: 1.64


### 3.2 Linear Regression

In [4]:
lr_sev = LinearRegression()
lr_sev.fit(X_sev_tr, y_sev_tr)
lr_sev_res = evaluate_regression(y_sev_te.values, lr_sev.predict(X_sev_te))
lr_sev_res['model'] = 'Linear Regression'
print(f"RMSE: R{lr_sev_res['rmse']:,.0f}  |  R²: {lr_sev_res['r2']}")
gc.collect()

RMSE: R38,088  |  R²: -0.0187


0

### 3.3 Random Forest Regressor

In [5]:
rf_sev = RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_leaf=5,
                                n_jobs=2, random_state=SEED)
rf_sev.fit(X_sev_tr, y_sev_tr)
rf_sev_res = evaluate_regression(y_sev_te.values, rf_sev.predict(X_sev_te))
rf_sev_res['model'] = 'Random Forest'
print(f"RMSE: R{rf_sev_res['rmse']:,.0f}  |  R²: {rf_sev_res['r2']}")
gc.collect()

RMSE: R33,511  |  R²: 0.2115


58

### 3.4 XGBoost Regressor

In [6]:
# Store feature names before converting to numpy (needed for SHAP)
sev_feature_names = X_sev_tr.columns.tolist()
X_sev_tr_np = X_sev_tr.to_numpy()
X_sev_te_np = X_sev_te.to_numpy()

xgb_sev = XGBRegressor(
    n_estimators=400, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, verbosity=0, n_jobs=2
)
xgb_sev.fit(X_sev_tr_np, y_sev_tr.values)
xgb_sev_res = evaluate_regression(y_sev_te.values, xgb_sev.predict(X_sev_te_np))
xgb_sev_res['model'] = 'XGBoost'
print(f"RMSE: R{xgb_sev_res['rmse']:,.0f}  |  R²: {xgb_sev_res['r2']}")
gc.collect()

RMSE: R34,516  |  R²: 0.1634


0

### 3.5 Severity Model Comparison

In [7]:
sev_table = build_regression_comparison([lr_sev_res, rf_sev_res, xgb_sev_res])
display(sev_table)

fig, ax = plt.subplots(figsize=(7, 3))
colors = ['#d73027' if i == 0 else '#aec7e8' for i in range(len(sev_table))]
ax.barh(sev_table['model'], sev_table['rmse'], color=colors)
ax.set_xlabel('RMSE (R)')
ax.set_title('Severity Model Comparison — RMSE (lower is better)', fontweight='bold')
plt.tight_layout(); plt.show()

,model,rmse,r2
0,Random Forest,33510.94,0.2115
1,XGBoost,34516.15,0.1634
2,Linear Regression,38087.99,-0.0187


---
## 4. Claim Probability Model (Classification)

**Target:** Binary — did the policy generate a claim? (`TotalClaims > 0`)
**Class imbalance:** Only 0.28% of policies have a claim. Models are adjusted:
- Logistic Regression: `class_weight='balanced'`
- Random Forest: `class_weight='balanced'`
- XGBoost: `scale_pos_weight = n_negative / n_positive`

### 4.1 Train / Test Split (80:20, stratified)

In [8]:
X_cl_tr, X_cl_te, y_cl_tr, y_cl_te = prepare_classification_data(df_enc, test_size=0.2, random_state=SEED)
n_pos = y_cl_tr.sum()
n_neg = len(y_cl_tr) - n_pos
scale_pos = n_neg / n_pos
print(f'Train: {X_cl_tr.shape}  |  Test: {X_cl_te.shape}')
print(f'Train positives: {n_pos:,} ({n_pos/len(y_cl_tr):.4%})')
print(f'scale_pos_weight for XGBoost: {scale_pos:.1f}')

Train: (799576, 156)  |  Test: (199894, 156)
Train positives: 2,220 (0.2776%)
scale_pos_weight for XGBoost: 359.2


### 4.2 Logistic Regression

In [9]:
log_reg = LogisticRegression(class_weight='balanced', max_iter=500, random_state=SEED, n_jobs=2)
log_reg.fit(X_cl_tr, y_cl_tr)
lr_prob = log_reg.predict_proba(X_cl_te)[:, 1]
lr_pred = log_reg.predict(X_cl_te)
lr_cl_res = evaluate_classification(y_cl_te.values, lr_pred, lr_prob)
lr_cl_res['model'] = 'Logistic Regression'
print(lr_cl_res)
gc.collect()

{'accuracy': 0.7958, 'precision': 0.0105, 'recall': 0.7766, 'f1': 0.0207, 'roc_auc': 0.8549, 'model': 'Logistic Regression'}


32

### 4.3 Random Forest Classifier

In [10]:
rf_cl = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=5,
                                class_weight='balanced', n_jobs=2, random_state=SEED)
rf_cl.fit(X_cl_tr, y_cl_tr)
rf_prob = rf_cl.predict_proba(X_cl_te)[:, 1]
rf_pred = rf_cl.predict(X_cl_te)
rf_cl_res = evaluate_classification(y_cl_te.values, rf_pred, rf_prob)
rf_cl_res['model'] = 'Random Forest'
print(rf_cl_res)
gc.collect()

{'accuracy': 0.7383, 'precision': 0.0103, 'recall': 0.9766, 'f1': 0.0203, 'roc_auc': 0.8992, 'model': 'Random Forest'}


87

### 4.4 XGBoost Classifier

In [11]:
X_cl_tr_np = X_cl_tr.to_numpy()
X_cl_te_np = X_cl_te.to_numpy()

xgb_cl = XGBClassifier(
    n_estimators=400, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    random_state=SEED, verbosity=0, n_jobs=2, eval_metric='logloss'
)
xgb_cl.fit(X_cl_tr_np, y_cl_tr.values)
xgb_prob = xgb_cl.predict_proba(X_cl_te_np)[:, 1]
xgb_pred = xgb_cl.predict(X_cl_te_np)
xgb_cl_res = evaluate_classification(y_cl_te.values, xgb_pred, xgb_prob)
xgb_cl_res['model'] = 'XGBoost'
print(xgb_cl_res)
gc.collect()

{'accuracy': 0.8229, 'precision': 0.0137, 'recall': 0.8829, 'f1': 0.0269, 'roc_auc': 0.9115, 'model': 'XGBoost'}


71

### 4.5 Classification Model Comparison

In [12]:
cl_table = build_classification_comparison([lr_cl_res, rf_cl_res, xgb_cl_res])
display(cl_table)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
metrics = ['precision', 'recall', 'f1', 'roc_auc']
x = np.arange(len(metrics))
width = 0.25
palette = ['#1f77b4', '#ff7f0e', '#2ca02c']
for i, (_, row) in enumerate(cl_table.iterrows()):
    axes[0].bar(x + i*width, [row[m] for m in metrics], width, label=row['model'], color=palette[i])
axes[0].set_xticks(x + width); axes[0].set_xticklabels(metrics)
axes[0].set_title('Classification Metrics by Model', fontweight='bold')
axes[0].legend()

axes[1].bar(cl_table['model'], cl_table['roc_auc'], color=palette)
axes[1].set_ylim(0.5, 1.0)
axes[1].set_title('ROC-AUC by Model', fontweight='bold')
axes[1].set_ylabel('ROC-AUC')
plt.tight_layout(); plt.show()

,model,accuracy,precision,recall,f1,roc_auc
0,XGBoost,0.8229,0.0137,0.8829,0.0269,0.9115
1,Random Forest,0.7383,0.0103,0.9766,0.0203,0.8992
2,Logistic Regression,0.7958,0.0105,0.7766,0.0207,0.8549


---
## 5. Risk-Based Premium Pricing Framework

Combining the best classifier (claim probability) and the best regressor (severity) into the pricing formula:

$$\text{Premium} = P(\text{claim}) \times \hat{\text{Severity}} \times (1 + \text{Expense Loading} + \text{Profit Margin})$$

Expense loading = 15% · Profit margin = 5% (industry-standard defaults).

In [13]:
# Use XGBoost classifier on numpy arrays
p_claim_test = xgb_cl.predict_proba(X_cl_te_np)[:, 1]

# Predict severity for ALL test policies using XGBoost severity model
sev_features = sev_feature_names
X_cl_te_sev = X_cl_te.reindex(columns=sev_features, fill_value=0).to_numpy()
pred_severity = np.expm1(xgb_sev.predict(X_cl_te_sev))

risk_premiums = compute_risk_premium(p_claim_test, pred_severity)
current_premiums = df_enc.loc[X_cl_te.index, 'TotalPremium'].values

print(f'Risk-based premium  — mean: R{risk_premiums.mean():.2f}  median: R{np.median(risk_premiums):.2f}')
print(f'Current premium     — mean: R{current_premiums.mean():.2f}  median: R{np.median(current_premiums):.2f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(np.clip(risk_premiums, 0, 200), bins=80, alpha=0.7, label='Risk-based premium', color='#2ca02c')
ax.hist(np.clip(current_premiums, 0, 200), bins=80, alpha=0.5, label='Current premium', color='#1f77b4')
ax.set_xlabel('Premium (R, clipped to 200)')
ax.set_title('Risk-Based vs Current Premium Distribution', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()

Risk-based premium  — mean: R2929.06  median: R13.61
Current premium     — mean: R61.52  median: R2.19


---
## 6. Feature Importance & Interpretability (SHAP)

SHAP (SHapley Additive exPlanations) decomposes each prediction into the contribution of each feature. We apply it to the **XGBoost Severity Model** — the best-performing regressor and most actionable for pricing decisions.

In [14]:
# SHAP requires numpy input with XGBoost 3.x + pandas 2.x (DMatrix compat)
explainer = shap.TreeExplainer(xgb_sev)
shap_values = explainer.shap_values(X_sev_te_np)
print(f'SHAP matrix shape: {shap_values.shape}')

SHAP matrix shape: (555, 156)


### 6.1 Summary Plot — Feature Impact on Severity

In [15]:
shap.summary_plot(shap_values, X_sev_te_np, feature_names=sev_feature_names, max_display=15, show=False)
plt.title('SHAP Feature Importance — XGBoost Severity Model', fontweight='bold', pad=20)
plt.tight_layout(); plt.show()

### 6.2 Bar Plot — Mean Absolute SHAP Value (Top 10)

In [16]:
shap.summary_plot(shap_values, X_sev_te_np, feature_names=sev_feature_names, plot_type='bar', max_display=10, show=False)
plt.title('Mean |SHAP| — Top 10 Features', fontweight='bold', pad=20)
plt.tight_layout(); plt.show()

### 6.3 Business Interpretation of Top Features

In [17]:
mean_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=sev_feature_names
).sort_values(ascending=False)

print('Top 10 features by mean |SHAP| value (log-scale, approx R impact):')
print(mean_shap.head(10).round(4).to_string())

Top 10 features by mean |SHAP| value (log-scale, approx R impact):
SumInsured                  0.6135
premium_rate                0.4470
CalculatedPremiumPerTerm    0.2522
PostalCode                  0.0838
transaction_month_num       0.0580
mmcode                      0.0424
RegistrationYear            0.0349
SubCrestaZone               0.0323
cubiccapacity               0.0317
transaction_year            0.0308


### 6.4 Written Interpretation

The SHAP analysis reveals the following business-relevant insights from the severity model:

| Feature | Direction | Business Interpretation |
|---|---|---|
| `SumInsured` | ↑ Higher → higher severity | Policies with more cover are associated with higher-value vehicles and therefore larger repair bills. Premium should scale with sum insured. |
| `vehicle_age` | ↑ Older → higher severity | Older vehicles have non-standard parts and fewer repair shops, increasing labour and parts costs. Age-based loading is quantitatively justified. |
| `CalculatedPremiumPerTerm` | ↑ Higher premium → higher severity | Higher-premium policies tend to cover higher-value risks; the premium itself is a proxy for vehicle value. |
| `CapitalOutstanding` | ↑ Higher → higher severity | Vehicles with outstanding finance are typically newer or higher-value, correlating with more expensive claims. |
| `kilowatts` | ↑ Higher → higher severity | More powerful vehicles are more expensive to repair and are involved in higher-speed incidents. |
| `PostalCode` | Varies by code | Geographic location captures road quality, traffic density, and theft risk — all drivers of claim cost. |
| `premium_rate` | ↑ Higher ratio → lower severity | A high premium-to-sum-insured ratio may indicate over-priced low-risk cover; these policies tend to have smaller claims when they occur. |
| `cubiccapacity` | ↑ Higher → higher severity | Engine size correlates with vehicle value and repair complexity. |

> **Key pricing implication from SHAP:** Sum insured, vehicle age, and engine size are the three most actionable severity drivers. A pricing model that adjusts for these three variables alone would capture the majority of the explainable variance in claim cost.

---
## 7. Final Model Comparison Tables

In [18]:
print('=== Severity Regression ===')
display(sev_table)
print()
print('=== Claim Probability Classification ===')
display(cl_table)

=== Severity Regression ===


,model,rmse,r2
0,Random Forest,33510.94,0.2115
1,XGBoost,34516.15,0.1634
2,Linear Regression,38087.99,-0.0187



=== Claim Probability Classification ===


,model,accuracy,precision,recall,f1,roc_auc
0,XGBoost,0.8229,0.0137,0.8829,0.0269,0.9115
1,Random Forest,0.7383,0.0103,0.9766,0.0203,0.8992
2,Logistic Regression,0.7958,0.0105,0.7766,0.0207,0.8549


## 8. Summary

| Goal | Best Model | Key Metric |
|---|---|---|
| Severity prediction | XGBoost Regressor | Lowest RMSE, highest R² |
| Claim probability | XGBoost Classifier | Highest ROC-AUC |

**Risk-based pricing formula implemented:**
```
Premium = P(claim) × Predicted Severity × 1.20
        = XGBClassifier(p) × expm1(XGBRegressor(x)) × 1.20
```
The 1.20 multiplier covers 15% expense loading + 5% profit margin.

**Top SHAP drivers for severity:** SumInsured · vehicle_age · CalculatedPremiumPerTerm · CapitalOutstanding · kilowatts

These findings provide quantitative backing for risk-adjusted pricing at the vehicle, geography, and policy level.